# SegNet: phân đoạn bàn tay trên ảnh độ sâu (PyTorch)

Bản PyTorch của `Yêu cầu/examples/5. Others/1_segnet.ipynb` (gốc dựa trên `imlab-uiip/keras-segnet`), giữ nguyên thứ tự mục và cấu hình của gốc.

- Vào: ảnh độ sâu (Kinect) 96×96, 1 kênh, giá trị 0–255 (không chuẩn hóa như gốc).
- Ra: nhãn cho **từng pixel**, 7 lớp: lòng bàn tay, 5 ngón, nền.
- Mô hình: encoder kiểu VGG-16 (13 conv, mỗi conv có BatchNorm + ReLU, 5 lần MaxPool) và decoder đối xứng dùng UpSampling, cuối là conv 1×1 ra 7 kênh + BatchNorm + softmax theo pixel.
- Huấn luyện: `categorical_crossentropy`, Adam lr 1e-4, batch 8, 15 epoch; validation là tập test như gốc.

| | Keras gốc (Keras 2.0.9, GTX 1080) | PyTorch |
|---|---|---|
| Số tham số | 29.458.211 tổng, 29.442.325 train được | kiểm bằng `assert` ở mục (2) |
| Pixel accuracy trên test | 0.7732 | xem mục (4) |
| Test loss | 0.6860 | xem mục (4) |
| Thời gian | 750 s (12,5 phút) | xem mục (3) |

# Import libraries

Keras gốc:
```python
from keras import models
from keras.layers import Conv2D, BatchNormalization, Activation, MaxPooling2D, UpSampling2D, Reshape, Permute
from keras.optimizers import Adam
import os, time, numpy as np, matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # để import pytorch/common.py
from common import *

import os
import torch.nn.functional as F
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize

# True: chỉ chạy thử (ít ảnh, 1 epoch) để kiểm code trên CPU; bản nộp phải là False (chạy GPU)
CHAY_THU = False

set_seed(0)
device = get_device()
TEN = "1_segnet"

In [ ]:
n_classes = 7
kernel = 3
img_h = img_w = 96

# (1) Data

Keras gốc (rút gọn):
```python
def prepare_segnet_data(root_path):
    loi = os.listdir(root_path)
    data_x = np.zeros((int(len(loi)), img_h, img_w, 1))
    data_y = np.zeros((int(len(loi)), img_h, img_w, n_classes), dtype=np.uint8)
    for im_ind, fn in enumerate(loi):
        def gray2classes(label_img, ...):   # mức xám -> one-hot 7 lớp
            # palm 255 -> 0, finger 1..5: 43, 85, 128, 170, 212 -> 1..5, background 0 -> 6
        fn = os.path.join(root_path, fn)
        labels_fn = fn.replace('depth', 'labels')
        img = np.uint8(resize(imread(fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))
        label = gray2classes(np.uint8(resize(imread(labels_fn), (img_h, img_w), order=0, mode='constant', preserve_range=True)))
        ...
    data_y = data_y.reshape((int(len(loi)), img_h * img_w, n_classes))
    return data_x, data_y
```

Khác Keras:
- **Đường dẫn nhãn**: gốc dùng `fn.replace('depth', 'labels')`, thay **mọi** chữ "depth" trong đường dẫn. Đường dẫn trong repo đi qua thư mục `depth_hand/` nên cách đó sẽ hỏng (`labels_hand/`). Ở đây chỉ đổi tên thư mục cuối và tên file.
- Đọc bằng `sorted(os.listdir(...))` để thứ tự giống nhau trên mọi máy (trên máy này `os.listdir` cũng ra đúng thứ tự đó).
- Resize giữ nguyên như gốc (`order=0`: lấy điểm gần nhất, nên không sinh mức xám mới).
- Nhãn lưu dạng **chỉ số lớp** `(N, 96, 96)` thay one-hot `(N, 9216, 7)`, vì `nn.CrossEntropyLoss` nhận chỉ số lớp và tính được trực tiếp trên ảnh `(N, 7, 96, 96)`.
- **Pixel lạ**: ảnh nhãn có thêm 4 mức xám 51, 102, 153, 204 (khoảng 0,2% số pixel train, 0,4% test) không thuộc 7 lớp. Ở gốc các pixel này thành one-hot toàn 0: góp 0 vào loss (nhưng vẫn tính trong mẫu số khi lấy trung bình), và khi tính accuracy thì `argmax` của vector 0 là lớp 0 (palm). Ở đây gán nhãn −100 cho các pixel này rồi xử lý đúng như vậy ở mục (3).

In [ ]:
MAU_LOP = [255, 43, 85, 128, 170, 212, 0]     # chỉ số lớp 0..6 = palm, finger 1..5, background
BO_QUA = -100                                 # pixel không thuộc 7 mức xám trên


def gray2classes(label_img):
    """Ảnh nhãn mức xám (96, 96) -> chỉ số lớp (96, 96); pixel lạ = -100."""
    out = np.full(label_img.shape, BO_QUA, dtype=np.int64)
    for class_indx, class_color in enumerate(MAU_LOP):
        out[label_img == class_color] = class_indx
    return out


def prepare_segnet_data(root_path, n=None):
    root_path = Path(root_path)
    loi = sorted(os.listdir(root_path))[:n]
    data_x = np.zeros((len(loi), 1, img_h, img_w), dtype=np.float32)    # NCHW, 0–255
    data_y = np.zeros((len(loi), img_h, img_w), dtype=np.int64)
    for im_ind, fn in enumerate(loi):
        labels_fn = root_path.parent / "labels" / fn.replace('depth', 'labels')
        img = np.uint8(resize(imread(root_path / fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))
        lab = np.uint8(resize(imread(labels_fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))
        data_x[im_ind, 0] = img
        data_y[im_ind] = gray2classes(lab)
    return torch.from_numpy(data_x), torch.from_numpy(data_y)


GOC = EXAMPLES_DIR / "5. Others" / "depth_hand" / "data" / "segnet"
x_train, y_train = prepare_segnet_data(GOC / "train" / "depth", 40 if CHAY_THU else None)
x_test, y_test = prepare_segnet_data(GOC / "test" / "depth", 1050 if not CHAY_THU else 16)
print("train", tuple(x_train.shape), tuple(y_train.shape), "| test", tuple(x_test.shape), tuple(y_test.shape))
print("pixel lạ: train %.4f%%, test %.4f%%" % (100 * (y_train == BO_QUA).float().mean(), 100 * (y_test == BO_QUA).float().mean()))

In [ ]:
# Như gốc: ảnh test số 1000 và 7 bản đồ lớp của nó
k = min(1000, len(x_test) - 1)
plt.imshow(x_test[k, 0], cmap='gray')
plt.show()

plt.figure(figsize=(18, 5))
for fin_ind in range(1, n_classes + 1):
    plt.subplot(1, 7, fin_ind)
    plt.imshow((y_test[k] == fin_ind - 1).numpy(), cmap='gray')
plt.show()

# (2) Declare model

Keras gốc (rút gọn, đầy đủ trong notebook gốc):
```python
encoding_layers = [
    Conv2D(64, (kernel, kernel), padding='same', input_shape=(img_h, img_w, 1)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, (kernel, kernel), padding='same'), BatchNormalization(), Activation('relu'),
    MaxPooling2D(),
    # 128 x2, MaxPool; 256 x3, MaxPool; 512 x3, MaxPool; 512 x3, MaxPool
]
decoding_layers = [
    UpSampling2D(),
    # 512 x3; UpSampling; 512, 512, 256; UpSampling; 256, 256, 128; UpSampling; 128, 64; UpSampling; 64
    Conv2D(n_classes, (1, 1), padding='valid'),
    BatchNormalization(),
]
autoencoder.add(Reshape((n_classes, img_h * img_w)))
autoencoder.add(Permute((2, 1)))
autoencoder.add(Activation('softmax'))
```

Khác Keras:
- Mỗi bộ `Conv2D` + `BatchNormalization` + `Activation('relu')` gom vào hàm `cbr`. Thứ tự lớp giống hệt gốc.
- **BatchNorm**: `nn.BatchNorm2d(c, eps=1e-3, momentum=0.01)`. Keras mặc định `epsilon=1e-3`, `momentum=0.99` (giữ 99% giá trị cũ); `momentum` của PyTorch định nghĩa ngược (phần giá trị mới), nên 0.99 của Keras là 0.01 của PyTorch. PyTorch mặc định `eps=1e-5`, `momentum=0.1`, nên phải đặt lại.
- **Số tham số**: Keras "Total params" tính cả running mean/var của BatchNorm (15.886 "Non-trainable"). PyTorch để chúng là buffer, không nằm trong `parameters()`, nên so `count_params` với "Trainable params" 29.442.325, và `count_all` (tham số + buffer) với tổng 29.458.211.
- **Đầu ra**: bỏ `Reshape` + `Permute` + softmax. Mô hình trả logits `(N, 7, 96, 96)`; `nn.CrossEntropyLoss` tính softmax theo chiều lớp cho từng pixel, giống softmax trên `(N, 9216, 7)` của gốc.
- `UpSampling2D()` = `nn.Upsample(scale_factor=2, mode='nearest')`. Kích thước: 96 → 48 → 24 → 12 → 6 → 3 rồi ngược lại 3 → 6 → … → 96.

In [ ]:
def cbr(c_in, c_out, k=kernel):
    """Conv2D same + BatchNormalization + ReLU như một bộ trong gốc."""
    return [nn.Conv2d(c_in, c_out, k, padding=k // 2),
            nn.BatchNorm2d(c_out, eps=1e-3, momentum=0.01),   # = Keras epsilon 1e-3, momentum 0.99
            nn.ReLU()]


class SegNet(nn.Module):
    def __init__(self, n_classes=7):
        super().__init__()
        up = lambda: nn.Upsample(scale_factor=2, mode='nearest')    # UpSampling2D()
        pool = lambda: nn.MaxPool2d(2)                              # MaxPooling2D()
        self.encoder = nn.Sequential(
            *cbr(1, 64), *cbr(64, 64), pool(),                          # (N, 64, 48, 48)
            *cbr(64, 128), *cbr(128, 128), pool(),                      # (N, 128, 24, 24)
            *cbr(128, 256), *cbr(256, 256), *cbr(256, 256), pool(),     # (N, 256, 12, 12)
            *cbr(256, 512), *cbr(512, 512), *cbr(512, 512), pool(),     # (N, 512, 6, 6)
            *cbr(512, 512), *cbr(512, 512), *cbr(512, 512), pool(),     # (N, 512, 3, 3)
        )
        self.decoder = nn.Sequential(
            up(), *cbr(512, 512), *cbr(512, 512), *cbr(512, 512),       # (N, 512, 6, 6)
            up(), *cbr(512, 512), *cbr(512, 512), *cbr(512, 256),       # (N, 256, 12, 12)
            up(), *cbr(256, 256), *cbr(256, 256), *cbr(256, 128),       # (N, 128, 24, 24)
            up(), *cbr(128, 128), *cbr(128, 64),                        # (N, 64, 48, 48)
            up(), *cbr(64, 64),                                         # (N, 64, 96, 96)
            nn.Conv2d(64, n_classes, 1),                                # Conv2D(n_classes, (1, 1), 'valid')
            nn.BatchNorm2d(n_classes, eps=1e-3, momentum=0.01),         # (N, 7, 96, 96), logits
        )

    def forward(self, x):                     # (N, 1, 96, 96), giá trị 0–255
        return self.decoder(self.encoder(x))  # (N, 7, 96, 96): softmax nằm trong CrossEntropyLoss


model = init_keras_like(SegNet(n_classes)).to(device)
summary(model, input_size=(1, 1, img_h, img_w), depth=1)

In [ ]:
n_params = count_params(model)
n_all = count_all(model)
print("Train được: %d (Keras Trainable params: 29442325)" % n_params)
print("Tổng gồm running mean/var: %d (Keras Total params: 29458211)" % n_all)
assert n_params == 29_442_325
assert n_all == 29_458_211

# (3) Train defined model

Keras gốc:
```python
lr = 0.0001
batch_size = 8
nb_epoch = 15
optimizer = Adam(lr=lr)
autoencoder.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=['accuracy'])
autoencoder.fit(x_train, y_train, validation_data=(x_test, y_test),
                batch_size=batch_size, epochs=nb_epoch, verbose=1)
```

Loss và accuracy viết riêng để giống Keras ở các pixel lạ (nhãn −100):
- `loss_keras`: cross-entropy từng pixel, pixel lạ góp 0, rồi lấy trung bình trên **mọi** pixel (kể cả pixel lạ), như `categorical_crossentropy` với one-hot toàn 0.
- `pixel_accuracy`: pixel lạ được tính là lớp 0 (palm), như `argmax` của vector 0 trong Keras.

In [ ]:
def loss_keras(output, target):
    per_pixel = F.cross_entropy(output, target, ignore_index=BO_QUA, reduction='none')   # pixel lạ = 0
    return per_pixel.mean()


def pixel_accuracy(output, target):
    return accuracy(output, target.clamp(min=0))     # -100 -> 0 (palm), như Keras


lr = 0.0001
batch_size = 8
nb_epoch = 1 if CHAY_THU else 15
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_loader = make_loader(x_train, y_train, batch_size=batch_size, shuffle=True)
test_loader = make_loader(x_test, y_test, batch_size=batch_size, shuffle=False)

start_time = time.time()
history = fit(model, train_loader, test_loader, loss_keras, optimizer, nb_epoch, metric=pixel_accuracy)
training_time = time.time() - start_time
print('> training time: ' + str(training_time) + 's')

In [ ]:
fig = plot_history(history, None if CHAY_THU else TEN, metric_label="pixel accuracy",
                   title="SegNet (PyTorch)")
plt.show()

# (4) Evaluate trained model

Kết quả gốc: `Test loss: 0.6860`, `Test accuracy: 0.7732` (pixel accuracy, trung bình trên mọi pixel của 1050 ảnh test).

In [ ]:
score = evaluate(model, test_loader, loss_keras, pixel_accuracy)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
if not CHAY_THU:
    save_history(history, TEN)
    save_metrics(TEN, test_loss=score[0], test_metric=score[1], n_params=n_params, n_params_tong=n_all,
                 epochs_ran=len(history["loss"]), best_epoch=None, train_time_s=training_time,
                 device=device_name(device), seed=0)
    ghi_so_sanh(TEN, "5. Others", "pixel_acc", keras=0.7732, pytorch=score[1],
                keras_params=29442325, pytorch_params=n_params, epochs_keras=15, epochs_pytorch=nb_epoch,
                optimizer="Adam", lr=lr, batch=batch_size, device=device_name(device),
                train_time_s=training_time, seed=0,
                ghi_chu="So tham số train được (Keras tổng 29458211 gồm running mean/var BN); "
                        "BN eps 1e-3 momentum 0.01; pixel lạ xử lý như one-hot toàn 0 của Keras")

## Hiển thị kết quả phân đoạn

`class_to_rgb` giữ như gốc, chỉ đổi `np.float` (đã bị bỏ khỏi numpy mới) thành `float`.

In [ ]:
def class_to_rgb(output_dis):
    rgb_out = np.ones((img_h, img_w, 3), dtype=float)

    rgb_out[np.where(output_dis == 0)] = [0, 0, 0]
    rgb_out[np.where(output_dis == 1)] = [0, 0, 0.5]
    rgb_out[np.where(output_dis == 2)] = [0, 0.5, 0]
    rgb_out[np.where(output_dis == 3)] = [0.5, 0, 0]
    rgb_out[np.where(output_dis == 4)] = [0.5, 0, 0.5]
    rgb_out[np.where(output_dis == 5)] = [0.5, 0.5, 0]
    rgb_out[np.where(output_dis == 6)] = [1, 1, 1]

    return rgb_out

In [ ]:
# Gốc vẽ ảnh vào và kết quả dự đoán; thêm cột nhãn thật để so
ran_ind = np.random.choice(range(int(x_test.shape[0])), 1)[0]
segnet_out = predict(model, x_test[ran_ind:ran_ind + 1])           # (1, 7, 96, 96)
segnet_rgb = class_to_rgb(segnet_out[0].argmax(dim=0).numpy())     # gốc: argmax(reshape(96, 96, 7), axis=2)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.imshow(x_test[ran_ind, 0], cmap='gray'); plt.title('ảnh độ sâu')
plt.subplot(1, 3, 2); plt.imshow(segnet_rgb); plt.title('SegNet dự đoán')
plt.subplot(1, 3, 3); plt.imshow(class_to_rgb(y_test[ran_ind].clamp(min=0).numpy())); plt.title('nhãn thật')
if not CHAY_THU:
    plt.savefig(results_dir(TEN) / "vi_du_phan_doan.png", dpi=120, bbox_inches="tight")
plt.show()